In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy import stats

pd.set_option('display.float_format', '{:.2f}'.format)

###Load the Data

In [12]:
df = pd.read_csv('../data/student-mat.csv')

# Select correct columns using exact names from your dataset
df = df[['weekly_studytime', 'failures', 'absences', 
         'G1', 'G2', 'G3']].dropna()

# Rename for clarity
df.columns = ['StudyTime', 'Failures', 'Absences', 
              'Grade1', 'Grade2', 'FinalGrade']

print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

Shape: (395, 6)

First 5 rows:


,StudyTime,Failures,Absences,Grade1,Grade2,FinalGrade
0,2,0,6,5,6,6
1,2,0,4,5,5,6
2,2,3,10,7,8,10
3,3,0,2,15,14,15
4,2,0,4,6,10,10


In [17]:
X_simple = df[['Grade2']]
y = df['FinalGrade']

reg_simple = LinearRegression()
reg_simple.fit(x_simple,y)

print("=== Simple Linear Regression (Sklearn) ===")
print(f"Intercept (β₀):  {round(reg_simple.intercept_, 2)}")
print(f"Coefficient (β₁): {round(reg_simple.coef_[0], 2)}")
print(f"R-squared:        {round(reg_simple.score(X_simple, y), 4)}")
print(f"\nInterpretation: For every 1 point increase in Grade2,")
print(f"Final Grade increases by {round(reg_simple.coef_[0], 2)} points")

=== Simple Linear Regression (Sklearn) ===
Intercept (β₀):  -1.39
Coefficient (β₁): 1.1
R-squared:        0.8188

Interpretation: For every 1 point increase in Grade2,
Final Grade increases by 1.1 points


In [18]:
# Sklearn does not give p-values by default
# We calculate them manually using scipy

X_simple_const = np.column_stack([np.ones(len(X_simple)), X_simple])
n = len(y)
k = X_simple_const.shape[1]

y_pred = reg_simple.predict(X_simple)
residuals = y - y_pred
RSS = np.sum(residuals**2)
SE = np.sqrt(RSS / (n - k))

# Standard errors and t-stats
XtX_inv = np.linalg.inv(X_simple_const.T @ X_simple_const)
se_coeffs = SE * np.sqrt(np.diag(XtX_inv))
t_stats = np.array([reg_simple.intercept_] + 
                    list(reg_simple.coef_)) / se_coeffs
p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n-k))

summary = pd.DataFrame({
    'Coefficient': [round(reg_simple.intercept_, 4)] + 
                   [round(c, 4) for c in reg_simple.coef_],
    'Std Error':   [round(s, 4) for s in se_coeffs],
    'T-statistic': [round(t, 4) for t in t_stats],
    'P-value':     [round(p, 4) for p in p_values],
    'Significant': ['Yes' if p < 0.05 else 'No' for p in p_values]
}, index=['Intercept', 'Grade2'])

print("=== Stats Model Like Summary Table ===")
print(summary)

=== Stats Model Like Summary Table ===
           Coefficient  Std Error  T-statistic  P-value Significant
Intercept        -1.39       0.30        -4.69     0.00         Yes
Grade2            1.10       0.03        42.14     0.00         Yes


In [19]:
# Without normalization StudyTime (1-4) and Absences (0-93)
# are on completely different scales
# The model unfairly gives more weight to larger numbers

X_multi = df[['StudyTime', 'Failures', 'Absences', 'Grade1', 'Grade2']]
y = df['FinalGrade']

# Before normalization
print("Before normalization:")
print(X_multi.describe().loc[['min', 'max', 'mean']])

# Apply StandardScaler — transforms each column to mean=0, std=1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_multi)
X_scaled = pd.DataFrame(X_scaled, columns=X_multi.columns)

print("\nAfter normalization:")
print(X_scaled.describe().loc[['min', 'max', 'mean']].round(2))
print("\nNow all variables are on the same scale — fair comparison")

Before normalization:
      StudyTime  Failures  Absences  Grade1  Grade2
min        1.00      0.00      0.00    3.00    0.00
max        4.00      3.00     75.00   19.00   19.00
mean       2.04      0.33      5.71   10.91   10.71

After normalization:
      StudyTime  Failures  Absences  Grade1  Grade2
min       -1.24     -0.45     -0.71   -2.39   -2.85
max        2.34      3.59      8.67    2.44    2.21
mean      -0.00     -0.00     -0.00    0.00   -0.00

Now all variables are on the same scale — fair comparison


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

reg_multi = LinearRegression()
reg_multi.fit(X_train, y_train)

y_pred = reg_multi.predict(X_test)
r2 = r2_score(y_test, y_pred)

print("=== Multiple Linear Regression (Sklearn) ===")
print(f"\nR-squared: {round(r2, 4)}")

print("\nFeature Coefficients (after normalization):")
coef_df = pd.DataFrame({
    'Feature': X_multi.columns,
    'Coefficient': reg_multi.coef_.round(4)
}).sort_values('Coefficient', ascending=False)
print(coef_df)

print("\nInterpretation:")
print("Larger coefficient = stronger influence on Final Grade")
print("Negative coefficient = this factor hurts final grade")

=== Multiple Linear Regression (Sklearn) ===

R-squared: 0.7822

Feature Coefficients (after normalization):
     Feature  Coefficient
4     Grade2         3.68
3     Grade1         0.48
2   Absences         0.31
0  StudyTime        -0.06
1   Failures        -0.34

Interpretation:
Larger coefficient = stronger influence on Final Grade
Negative coefficient = this factor hurts final grade


In [21]:
# Sklearn only gives R² — we calculate Adjusted R² manually
n = len(y_test)
k = X_scaled.shape[1]

adj_r2 = 1 - (1 - r2) * (n - 1) / (n - k - 1)

print("=== Model Evaluation ===")
print(f"R-squared:          {round(r2, 4)}")
print(f"Adjusted R-squared: {round(adj_r2, 4)}")
print(f"\nR² vs Adjusted R²: {round(r2 - adj_r2, 4)} difference")
print("Small difference = your extra variables genuinely helped")
print("Large difference = some variables are adding noise")

=== Model Evaluation ===
R-squared:          0.7822
Adjusted R-squared: 0.7673

R² vs Adjusted R²: 0.0149 difference
Small difference = your extra variables genuinely helped
Large difference = some variables are adding noise


In [22]:
# Which features actually matter?
# Compare R² with all features vs removing weakest ones

features_to_test = list(X_multi.columns)
results = []

for feature in features_to_test:
    remaining = [f for f in features_to_test if f != feature]
    X_temp = X_scaled[remaining]
    
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_temp, y, test_size=0.2, random_state=42
    )
    
    reg_temp = LinearRegression()
    reg_temp.fit(X_tr, y_tr)
    score = r2_score(y_te, reg_temp.predict(X_te))
    
    results.append({
        'Removed Feature': feature,
        'R² without it': round(score, 4),
        'Impact': round(score - r2, 4)
    })

results_df = pd.DataFrame(results).sort_values('Impact')
print("=== Feature Selection Analysis ===")
print(results_df)
print("\nNegative impact = removing this feature hurt the model")
print("Positive impact = this feature was adding noise — remove it")

=== Feature Selection Analysis ===
  Removed Feature  R² without it  Impact
4          Grade2           0.69   -0.09
3          Grade1           0.78   -0.00
0       StudyTime           0.78   -0.00
2        Absences           0.78   -0.00
1        Failures           0.80    0.01

Negative impact = removing this feature hurt the model
Positive impact = this feature was adding noise — remove it


In [23]:
# Predict final grade for a new student
new_student = pd.DataFrame({
    'StudyTime': [3],    # studies 3 hours/week
    'Failures':  [0],    # no past failures
    'Absences':  [2],    # 2 absences
    'Grade1':    [12],   # first period grade
    'Grade2':    [13]    # second period grade
})

# Must scale new data using the same scaler
new_student_scaled = scaler.transform(new_student)
predicted_grade = reg_multi.predict(new_student_scaled)[0]

print("=== Student Grade Prediction ===")
print(f"Study Time:  3 hrs/week")
print(f"Failures:    0")
print(f"Absences:    2")
print(f"Grade 1:     12/20")
print(f"Grade 2:     13/20")
print(f"\nPredicted Final Grade: {round(predicted_grade, 2)}/20")

=== Student Grade Prediction ===
Study Time:  3 hrs/week
Failures:    0
Absences:    2
Grade 1:     12/20
Grade 2:     13/20

Predicted Final Grade: 12.71/20


C:\Users\Asadullah Chandio\anaconda3\envs\ds_env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
